# Qdrant × Future AGI: Fixing RAG Decay

Live Qdrant vector search engine. Broken RAG app. Loop:

**Read the Future AGI dashboard → fix one thing in Qdrant → re-run the same 37 golden queries → read again.**

- **App**: fail · **Notebook**: fix · **Future AGI**: trace, judge, compare
- Projects: `pokedex-webinar`, `…-fix1-dedup`, `…-fix2-bge`, `…-fix3-hybrid`, `…-fix4-filter`
- Runner: `run_golden.py --stage iterN`
- Flaws are planted (skewed re-crawl duplicates, one stale chart, 110-char chunks); every measurement is real
- Judged scores live on the FI dashboard; section 7 holds the one recap table (offline fallback)

## 0 · Setup

Restore baseline (`uv run python snapshot.py restore` after any rehearsal). Retrieval: small dense model, no filter.

In [ ]:
import os, json, math
from pathlib import Path
from dotenv import load_dotenv
from qdrant_client import QdrantClient, models

import agent
from helpers import config, embeddings
from helpers.embeddings import dense, sparse, colbert

load_dotenv()
client = QdrantClient(url=os.environ["QDRANT_URL"], api_key=os.environ["QDRANT_API_KEY"])
COLLECTION = config.COLLECTION

# The golden set: 37 queries with expected answers and labeled correct documents.
rows = [json.loads(line) for line in Path("data/golden_dataset.jsonl").read_text().splitlines() if line.strip()]

# Start every run from the broken baseline: small dense model, no filter.
agent.set_retrieval(mode="minilm", current_only=False)
embeddings.warmup()   # load all models now, never during a live question
client.count(COLLECTION)

### Future AGI Tracing

Registered once in `agent.py`; traces from notebook, app, scripts:

```python
trace_provider = register(project_type=ProjectType.OBSERVE,
                          project_name=os.getenv("FI_PROJECT_NAME", "pokedex-rag"),
                          metadata={"stage": os.getenv("FI_STAGE_TAG", "adhoc")})
LangChainInstrumentor().instrument(tracer_provider=trace_provider)
```

- Root **"Pokedex Agent" span**: `input`, `output`, `retrieved_context` (every hop), `expected_answer` (golden runs)
- Map evals here
- **Answer-correctness judge** vs `expected_answer`: catches grounded-but-stale answers in section 5
- Qdrant searches: one retriever span each
- **Routing:** everything run live in this session — app questions and notebook asks — lands in **`pokedex-webinar-live`** (`FI_PROJECT_NAME` in `.env`). The five scored projects are written only by `run_golden.py --stage iterN`, which overrides the env per stage; those runs were executed before the session and stay frozen.

## 1 · The Pain: Retrieval Decayed As The Corpus Grew

Same queries, growing dex (33 retrieval-labeled; curve by `scaling_curve.py`): **recall@5 falls 0.67 → 0.39**. Code unchanged.

Baseline (`pokedex-webinar`):

- **Answer correctness 0.57**: wrong four times in ten; zero refusals. *"Does Steel resist Ghost and Dark?"* → **"Yes!"** (wrong)
- **2.0 searches per question**: re-searching hides weak retrieval (17 of 37 need 2+; one paraphrase takes five)

Context relevance reads a healthy-looking **0.92** while four questions in ten fail; groundedness (0.68) is the one early tell — and it can't say which layer is sick. **Lesson 1: a healthy retrieval score is not a healthy agent.**

In [ ]:
from IPython.display import Image
Image("data/scaling_curve.png")   # generated offline by scaling_curve.py

In [ ]:
# The scored 37-query baseline was run before the session -> project `pokedex-webinar`:
# !uv run python run_golden.py --stage iter1
# Live questions (app or notebook) land in `pokedex-webinar-live` and never touch the scored projects.

## 2 · Fix #1: Dedup The Collection

**Dashboard read:** open the low-relevance rows. Specimen: *"sleeps constantly and gains weight"* — `retrieved_context` is the same wrong Delcatty entry **× 4**; the agent still digs Snorlax out on a second search. **36 of 37 queries carry a duplicate in top-5.** Utilization (0.85) says nothing — copies of one chunk read as "used". Duplication in the data, not a model problem.

Fix: delete thousands of points live. `prep.py` snapshot ([snapshots](https://qdrant.tech/documentation/concepts/snapshots/)) restores in seconds.

In [ ]:
# Duplicate rate in the top-k before the fix: a slot is wasted if it repeats a fragment
# with the same doc_id and text we already retrieved.
def dup_rate(query):
    chunks = agent.retrieve(query, mode="minilm")
    keys = [(c["doc_id"], c["text"]) for c in chunks]
    return keys, 1 - len(set(keys)) / len(keys)

keys, rate = dup_rate("Tell me the Pokedex entry for Gengar")
print(f"duplicate rate: {rate:.0%}")
for k in keys: print(" ", k[0])

In [ ]:
# THE FIX: delete duplicate points with the same doc_id + chunk_index, and keep one copy.
# The small viz collection gets the same dedup so the Web UI point cloud thins out too.
# (Reversible: `uv run python snapshot.py restore` brings the broken baseline back.)
from helpers.dedup import find_duplicate_ids

for col in (COLLECTION, config.VIZ_COLLECTION):
    before = client.count(col).count
    dup_ids = find_duplicate_ids(client, col)
    client.delete(collection_name=col, points_selector=dup_ids)
    print(f"{col}: {before} -> {client.count(col).count} points ({len(dup_ids)} duplicates deleted)")

In [ ]:
keys, rate = dup_rate("Tell me the Pokedex entry for Gengar")
print(f"duplicate rate: {rate:.0%}")   # now distinct documents fill the top-k
for k in keys: print(" ", k[0])

In [ ]:
# LIVE: one question through the fixed collection — trace lands in `pokedex-webinar-live`.
answer, _ = agent.ask("Tell me the Pokedex entry for Gengar")
print(answer)   # before dedup: assembled from five copies of one doc; now: five distinct documents

# The scored run for this stage was executed before the session -> `pokedex-webinar-fix1-dedup`:
# !uv run python run_golden.py --stage iter2

Qdrant Web UI: `pokemon_viz` duplicate clusters thin out after dedup.

Exact copies gone. Near-duplicates remain (same-species flavor text across generations). Query-time fix: `group_by` or MMR for one result per document group.

**Re-measure** (`--stage iter2`, read `…-fix1-dedup`):

- Answer correctness **0.57 → 0.76** (failures 16 → 9) · duplicates in top-5 **36/37 → 2/37** · searches/question **2.00 → 1.46**
- Chunk attribution **0.89 → 1.00**; the Snorlax question now reads five distinct documents (the right one enters at rank 3)
- **Steel stops being wrong and starts hedging**: current chart reaches top-5, beside the stale one; trace metrics pass; answer-correctness stays red; section 5
- **App:** `Tell me the Pokedex entry for Gengar` — before: five copies of one doc; after: five distinct documents
- Surprise: chunk **utilization didn't move** (0.85 → 0.85) — five distinct chunks with one used scores the same as five copies of the used one. It never measured duplication

## 3 · Fix #2: Upgrade The Embedding Model

**Dashboard read:** nine failures left. Unnamed species descriptions retrieve lookalikes.

> *"the electric mouse Pokemon that stores electricity in the pouches on its cheeks"*
> retrieved: togedemaru, pichu, electrike, electrode above pikachu — the right entry scrapes in at **rank 5** → answer: **Pichu**, the higher-ranked lookalike (wrong)

The 384d model can't separate close neighbors at 1,025-species scale; at 151 species there were no clones. [Named vectors](https://qdrant.tech/documentation/manage-data/vectors/#named-vectors): offline backfill (`prep.py` — instant to add, slow to fill 23k points); live flip + A/B:

```python
client.create_vector_name(COLLECTION, "dense_strong", ...)   # offline, zero downtime
client.update_vectors(COLLECTION, points=[...])              # offline backfill
```

In [ ]:
# Point the agent at the candidate model — one line, instantly reversible: rollback
# is the same flip back to "minilm".
agent.set_retrieval(mode="bge")

# Both models are live on the SAME collection: four named vectors, one set of points.
info = client.get_collection(COLLECTION)
print("named vectors:", list(info.config.params.vectors), "+ sparse:", list(info.config.params.sparse_vectors))

# A/B both models on the golden queries this failure targets.
fix2 = [r for r in rows if r["exercises"] == "fix2_embedding"]

def recall_at_5(mode):
    """Share of queries with a correct document in the top 5 (unique docs, since
    fragments of one doc can fill several slots). Same scorer as verify_arc.py."""
    hits = 0
    for row in fix2:
        docs = []
        for chunk in agent.retrieve(row["query"], mode=mode, limit=30):
            if chunk["doc_id"] not in docs:
                docs.append(chunk["doc_id"])
        if set(docs[:5]) & set(row["gold_doc_ids"]):
            hits += 1
    return hits / len(fix2)

print(f"recall@5  small (MiniLM 384d)     = {recall_at_5('minilm'):.2f}")
print(f"recall@5  large (bge-large 1024d) = {recall_at_5('bge'):.2f}")

**0.64 → 1.00: keep bge**. Rollback: same one-line flip. Caveat for fix #3: these 14 queries target the class; slice score 1.00 means the class is fixed, not that production scores 1.00.

In [ ]:
# LIVE: the fix-2 signature question — trace lands in `pokedex-webinar-live`.
answer, _ = agent.ask("the electric mouse Pokemon that stores electricity in the pouches on its cheeks")
print(answer)   # post-dedup this answered Pichu (wrong lookalike); on bge it answers Pikachu

# The scored run for this stage was executed before the session -> `pokedex-webinar-fix2-bge`:
# !uv run python run_golden.py --stage iter3

**Re-measure** (`--stage iter3`, read `…-fix2-bge`):

- Answer correctness **0.76 → 0.92** (failures 9 → 3) · searches/question **1.46 → 1.24** · context relevance **0.98** · groundedness peaks at **0.92**
- **App:** `the electric mouse Pokemon that stores electricity in the pouches on its cheeks` — answer flips **Pichu → Pikachu**
- Nine queries outside the fix-2 slice get their gold doc into the top-5 too (eight of them fix-3 paraphrases)
- **Utilization stays flat (0.85 → 0.87)** through two retrieval fixes — it's a generator metric in a retrieval costume
- Steel confidently wrong again: stronger model ranks stale chart on top

## 4 · Fix #3: Hybrid Retrieval + Reranking

**Dashboard read:** nearly green (0.92). Underneath: sound-waves gold doc at **rank 12**; 4 of 18 gold docs below top-5; one missing from dense's top 30. Rewrites cover it (slice cost 2.4 searches/question at baseline). **Mis-ranking is invisible to answer metrics until the workaround stops working**. NDCG@5 0.61, MRR 0.55 against bge.

Fix: one [`query_points`](https://qdrant.tech/documentation/search/hybrid-queries/) call: dense + [miniCOIL](https://qdrant.tech/articles/minicoil/) → RRF → ColBERT rerank.

In [ ]:
q = "A Pokemon that throws skeletal weapons like a returning projectile to defeat opponents."

# Dense-only — even the strong model — loses this one completely: nothing in the query
# names Marowak, and the bone-boomerang paraphrase drowns among hundreds of battle
# entries. The gold doc is not even in dense's top 30.
dense_only = [c["doc_id"] for c in agent.retrieve(q, mode="bge")]
print("dense-only   :", dense_only)

# The hybrid + rerank call, spelled out. This is exactly what mode='hybrid' runs (agent.py).
dense_query = dense([q], config.MODEL_DENSE_STRONG, is_query=True)[0]
indices, values = sparse([q], is_query=True)[0]
colbert_query = colbert([q], is_query=True)[0]

candidates = models.Prefetch(
    prefetch=[
        models.Prefetch(query=dense_query, using=config.DENSE_STRONG,
                        limit=20, params=agent.EXACT),
        models.Prefetch(query=models.SparseVector(indices=indices, values=values),
                        using=config.SPARSE, limit=20),
    ],
    query=models.FusionQuery(fusion=models.Fusion.RRF),   # fuse dense + sparse lists
    limit=20,
)
hybrid = client.query_points(
    COLLECTION,
    prefetch=[candidates],
    query=colbert_query,          # rerank the 20 candidates with ColBERT
    using=config.COLBERT,
    limit=5,
    with_payload=["doc_id"],
).points
print("hybrid+rerank:", [h.payload["doc_id"] for h in hybrid])   # marowak-gen1-entry, rank 1

In [ ]:
# Flip the agent to hybrid, then score the queries dense retrieval was missing.
agent.set_retrieval(mode="hybrid")

fix3 = [r for r in rows if r["exercises"] == "fix3_hybrid"]

def score(mode):
    """recall@5, NDCG@5, MRR against the labeled documents (each query has one gold doc,
    so the ideal DCG is 1). Same scorer as verify_arc.py."""
    recall, ndcg, mrr = 0, 0, 0
    for row in fix3:
        docs = []
        for chunk in agent.retrieve(row["query"], mode=mode, limit=30):
            if chunk["doc_id"] not in docs:
                docs.append(chunk["doc_id"])
        top5, gold = docs[:5], set(row["gold_doc_ids"])
        recall += any(d in gold for d in top5)
        ndcg += sum(1 / math.log2(i + 2) for i, d in enumerate(top5) if d in gold)
        mrr += next((1 / (i + 1) for i, d in enumerate(top5) if d in gold), 0)
    n = len(fix3)
    return recall / n, ndcg / n, mrr / n

print("dense (bge)   recall / NDCG@5 / MRR: %.2f / %.2f / %.2f" % score("bge"))
print("hybrid+rerank recall / NDCG@5 / MRR: %.2f / %.2f / %.2f" % score("hybrid"))

**recall@5 0.78 → 0.89 · NDCG@5 0.61 → 0.77 · MRR 0.55 → 0.73.** Sparse recovers exact wording (Marowak: not in dense's top 30 → rank 1); rerank orders the pool. (Recall occasionally prints 0.94 — one gold doc flips rank 5↔7 on fusion ties. Single-query flips happen in either direction on ties; the slice score is the signal, don't re-run chasing one number.)

In [ ]:
# LIVE: the honest no-flip demo — trace lands in `pokedex-webinar-live`.
answer, _ = agent.ask("A Pokemon that lives in dark caves and uses sound waves to navigate and hunt.")
print(answer)   # several bats genuinely match; the win is in the panel/trace: real echolocation bats now, gold doc 12 -> 7

# The scored run for this stage was executed before the session -> `pokedex-webinar-fix3-hybrid`:
# !uv run python run_golden.py --stage iter4

**Re-measure** (`--stage iter4`, read `…-fix3-hybrid`): rank moves; answer barely does — it even **dips 0.92 → 0.86**: the agent was already rescuing mis-ranked docs with extra rewrites, so fixing rank shows up in rank metrics and search counts, not the answer column. Right docs now land where the model reads:

| query (user's wording) | gold rank, bge | after hybrid+rerank |
|---|---|---|
| "throws skeletal weapons like a returning projectile" | not in top 30 | **1** |
| "crawls up surfaces using adhesive feet" | 7 | **2** |
| "lives in dark caves, uses sound waves" | 12 | 7 — top-5 now fills with actual echolocation bats |
| "tumbles downhill, passes through barriers" | 11 | **5** (wobbles on fusion ties — some passes drop it out; tie noise, not signal) |

Searches/question: **1.24 → 1.14**. **No-hallucination hits its arc low (0.81 → 0.70)** with groundedness passing the flagged rows — confident ranking exposes generator embellishment.

**App:** `A Pokemon that lives in dark caves and uses sound waves to navigate and hunt.` — no answer flip; the panel fills with echolocation bats.

Steel: hybrid pulls **both** charts to ranks 1–2; agent hedges. Ranking surfaces the conflict; metadata resolves it.

## 5 · Close The Cold Open With Metadata

The same Steel question, every run so far:

| run | answer |
|---|---|
| baseline | "**Yes!**" — wrong; the stale Gen-5 chart fills every slot the model reads |
| dedup | "It **depends**…" — current chart finally visible, next to the stale one |
| bge | "**Yes**" — wrong; stale chart back on top |
| hybrid | "It **depends**…" — both charts, ranks 1 and 2 |

Four retrieval upgrades, zero correct answers. **Currency is not in the text — it lives in metadata.** Confidently-wrong runs score 1.00: grounded in the wrong document. One eval stays red: answer-correctness.

```text
Score 1 if the answer commits to the same core claim as the reference. Extra correct detail
is fine; a shorter answer is fine if it does not contradict the reference.
Score 0 if the answer contradicts the reference, refuses, or lists conflicting possibilities
without committing to the reference's claim as the current fact. The question asks for one
fact, so an answer that says it depends is not a correct answer.
```

Points carry `is_current`; index it and filter.

In [ ]:
q = "Does the Steel type resist Ghost and Dark attacks?"
before = [c["doc_id"] for c in agent.retrieve(q, mode="hybrid") if c["name"] == "steel"]
print("no filter :", before)

# THE FIX: index the is_current payload field, then have the agent filter on it.
client.create_payload_index(COLLECTION, "is_current", models.PayloadSchemaType.BOOL)
agent.set_retrieval(current_only=True)

after = [c["doc_id"] for c in agent.retrieve(q, mode="hybrid") if c["name"] == "steel"]
print("is_current:", after)

In [ ]:
# LIVE: ask the cold-open question again. Now it is grounded in the current document.
answer, _ = agent.ask("Does the Steel type resist Ghost and Dark attacks?")
print(answer)

# The scored run for this stage was executed before the session -> `pokedex-webinar-fix4-filter`:
# !uv run python run_golden.py --stage iter5

**Re-measure** (`--stage iter5`, read `…-fix4-filter`): the Steel arc ends —

> "Yes!" (wrong) → "it depends…" → "Yes" (wrong) → "it depends…" → **"No." (right)**

- Retrieved from `typechart-steel-gen6` alone; stale chart unreachable
- Judge goes green — its reasoning reads its own criterion back: the answer commits to the same core claim as the reference. A judge you can read is a judge you can trust
- Filter cost: no labeled document lost, every slice's retrieval unchanged, answer correctness lands at **0.92**

**App:** `Does the Steel type resist Ghost and Dark attacks?` — wrong "Yes" becomes correct "No"; the badge reads `current-only`.

Full before/after: Future AGI Experiments view; section 7 recap.

## 6 · Bonus: Multi-Hop In The Trace View

Compound question = multiple retriever spans under one root. Drowzee: hop one finds diet; later hops check matchup.

- **Multi-hop was also the agent's crutch**: baseline 2.0 searches/question; now 1.16
- Instrumentation: root `retrieved_context` needs **every** hop. Score hop one only and hallucination flags hop-two citations

In [ ]:
answer, chunks = agent.ask("What does Drowzee eat, and is that Pokemon weak to Bug-type attacks?")
print(answer)

## 7 · The Workflow You Take Home

Answer correctness **0.57 → 0.92**. Searches per question **2.00 → 1.16** (42% fewer retrieval loops). Dashboard read → small Qdrant fix → same queries. Offline fallback (live-account dashboard aggregates, runs of 2026-08-04; run files in `data/`):

| run | project | answer correctness | context relevance | chunk utilization | groundedness | chunk attribution | no-hallucination | searches/question |
|---|---|---|---|---|---|---|---|---|
| baseline | `pokedex-webinar` | **0.57** | 0.92 | 0.85 | 0.68 | 0.89 | 0.76 | 2.00 |
| fix 1 · dedup | `…-fix1-dedup` | **0.76** | 0.93 | 0.85 | 0.86 | 1.00 | 0.78 | 1.46 |
| fix 2 · bge | `…-fix2-bge` | **0.92** | 0.98 | 0.87 | 0.92 | 1.00 | 0.81 | 1.24 |
| fix 3 · hybrid | `…-fix3-hybrid` | **0.86** | 0.99 | 0.86 | 0.84 | 1.00 | 0.70 | 1.14 |
| fix 4 · filter | `…-fix4-filter` | **0.92** | 0.99 | 0.88 | 0.84 | 1.00 | 0.78 | 1.16 |

Lessons:

1. **A healthy retrieval score is not a healthy agent.** Baseline relevance read 0.92 while four questions in ten failed.
2. **Some metrics move the "wrong" way — or refuse to move.** Utilization stayed flat through four retrieval fixes (it measures the generator, and it never measured duplication); no-hallucination bottomed out right after the ranking fix (confident retrieval exposes embellishment); answer correctness itself dipped at hybrid — rank-sensitive gold-label metrics see what answer metrics can't.
3. **Grounded-in-the-wrong-document is invisible without ground truth.** Steel passed every trace metric while most wrong; only the judge stayed red, and only metadata fixed it.

Workflow: **measure → locate failing layer → fix it → re-run same queries → verify movement.**